In [1]:
from langchain_openai import ChatOpenAI

chatgpt = ChatOpenAI(model_name='gpt-4o-mini', temperature=0)

Linking Multiple Chains Sequentially in LCEL

Here we will see how we can link several LLM Chains sequentially using LCEL.

Typically the output from one chain might go as input into the next chain and so on.

The overall chain would run each chain sequentially in order till we get the final output which can be a combination of intermediate outputs and inputs from the previous chains

In [2]:
it_support_queue = [
    "I can't access my email. It keeps showing an error message. Please help.",
    "Tengo problemas con la VPN. No puedo conectarme a la red de la empresa. ¿Pueden ayudarme, por favor?",
    "Mon imprimante ne répond pas et n'imprime plus. J'ai besoin d'aide pour la réparer.",
    "我无法访问公司的网站。每次都显示错误信息。请帮忙解决。"
]

In [3]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

In [4]:
# Chain 1: Detect customer message language
prompt1 = """
  Act as a customer support agent.
  For the customer support message delimited below by triple backticks,
  Output the language of the message in one word only, e.g. Spanish

  Customer Message:
  ```{orig_msg}```
"""
prompt_template1 = ChatPromptTemplate.from_template(prompt1)
llm_chain1 = (prompt_template1
                  |
              chatgpt
                  |
              StrOutputParser())

In [5]:
llm_chain1.invoke({'orig_msg': it_support_queue[1]})

'Spanish'

In [13]:
from langchain_core.runnables import RunnablePassthrough, RunnableLambda

In [7]:
RunnablePassthrough.assign(orig_lang=llm_chain1).invoke({'orig_msg': it_support_queue[1]})

{'orig_msg': 'Tengo problemas con la VPN. No puedo conectarme a la red de la empresa. ¿Pueden ayudarme, por favor?',
 'orig_lang': 'Spanish'}

In [12]:
# Chain 2: Translate Customer Message to English
prompt2 = """
  Act as a customer support agent.
  For the customer message and customer message language delimited below by triple backticks,
  Translate the customer message from the customer message language to English
  if customer message language is not in English,
  else return back the original customer message.

  Customer Message:
  ```{orig_msg}```
  Customer Message Language:
  ```{orig_lang}```
"""
prompt_template2 = ChatPromptTemplate.from_template(prompt2)
llm_chain2 = (prompt_template2
                  |
              chatgpt
                  |
              StrOutputParser())

In [8]:
# Chain 3: Generate a resolution response in English
prompt3 = """
  Act as a customer support agent.
  For the customer support message delimited below by triple backticks,
  Generate an appropriate resolution response in English.

  Customer Message:
  ```{trans_msg}```
"""
prompt_template3 = ChatPromptTemplate.from_template(prompt3)
llm_chain3 = (prompt_template3
                  |
              chatgpt
                  |
              StrOutputParser())

In [9]:
# Chain 4: Translate resolution response from English to Customer's original language
prompt4 = """
  Act as a customer support agent.
  For the customer resolution response and target language delimited below by triple backticks,
  Translate the customer resolution response message from English to the target language
  if target language is not in English,
  else return back the original customer resolution response.

  Customer Resolution Response:
  ```{trans_response}```
  Target Language:
  ```{orig_lang}```
"""
prompt_template4 = ChatPromptTemplate.from_template(prompt4)
llm_chain4 = (prompt_template4
                  |
              chatgpt
                  |
              StrOutputParser())

In [14]:


final_chain = (
    RunnablePassthrough.assign(orig_lang=llm_chain1) #it will run the "llm_chain1" which will return the language of orig message and will be
      |                                              # assigned to orig_lang using RunnablePassthrough.assign
    RunnablePassthrough.assign(trans_msg=llm_chain2) #will take orig_lang and orig_message from 1st chain and return the translated message
      |                                              # which will be assign to trans_msg using RunnablePassthrough.assign
    RunnablePassthrough.assign(trans_response=llm_chain3) #will take trans_msg from 2st chain and and generate a resolution in engliash language and 
      |                                                   # will be assigned to trans_response using RunnablePassthrough.assign.
    RunnablePassthrough.assign(orig_response=llm_chain4) # it will take the english response from 3rd chain and orig_lang from 1st chain and translate
)                                                        # the response into original language. thai will be assigned to orig_response.

# NOTE: orig_msg, orig_lang, trans_msg, trans_response, orig_response can be used across this chain. and all of these will be available as part of this
# chain response.

In [15]:
response = final_chain.invoke({'orig_msg': it_support_queue[1]})

In [16]:
response

{'orig_msg': 'Tengo problemas con la VPN. No puedo conectarme a la red de la empresa. ¿Pueden ayudarme, por favor?',
 'orig_lang': 'Spanish',
 'trans_msg': "I am having problems with the VPN. I can't connect to the company's network. Can you please help me?",
 'trans_response': "Subject: Assistance with VPN Connection Issues\n\nDear [Customer's Name],\n\nThank you for reaching out to us regarding the issues you are experiencing with the VPN. I understand how important it is to connect to the company's network, and I'm here to help you resolve this matter.\n\nHere are a few troubleshooting steps you can try:\n\n1. **Check Your Internet Connection**: Ensure that you have a stable internet connection. You can try accessing a website to confirm this.\n\n2. **Restart the VPN Application**: Close the VPN application completely and then reopen it. Sometimes, a simple restart can resolve connection issues.\n\n3. **Verify Your Credentials**: Double-check that you are entering the correct userna

In [17]:
it_support_queue_formatted = [{'orig_msg': msg} for msg in it_support_queue]
it_support_queue_formatted

[{'orig_msg': "I can't access my email. It keeps showing an error message. Please help."},
 {'orig_msg': 'Tengo problemas con la VPN. No puedo conectarme a la red de la empresa. ¿Pueden ayudarme, por favor?'},
 {'orig_msg': "Mon imprimante ne répond pas et n'imprime plus. J'ai besoin d'aide pour la réparer."},
 {'orig_msg': '我无法访问公司的网站。每次都显示错误信息。请帮忙解决。'}]

In [18]:
responses = final_chain.map().invoke(it_support_queue_formatted)

In [19]:
responses

[{'orig_msg': "I can't access my email. It keeps showing an error message. Please help.",
  'orig_lang': 'English',
  'trans_msg': "I can't access my email. It keeps showing an error message. Please help.",
  'trans_response': "Subject: Assistance with Email Access\n\nDear [Customer's Name],\n\nThank you for reaching out to us. I’m sorry to hear that you’re having trouble accessing your email. I’d be happy to help you resolve this issue.\n\nTo better assist you, could you please provide me with the following information:\n1. The specific error message you are seeing.\n2. The email service you are using (e.g., Gmail, Yahoo, Outlook, etc.).\n3. Any troubleshooting steps you have already tried (e.g., resetting your password, clearing your browser cache, etc.).\n\nIn the meantime, here are a few general troubleshooting steps you can try:\n- Ensure that you are connected to the internet.\n- Try accessing your email from a different browser or device.\n- Clear your browser's cache and cookie

In [20]:
import pandas as pd
pd.DataFrame(responses)

,orig_msg,orig_lang,trans_msg,trans_response,orig_response
0,I can't access my email. It keeps showing an e...,English,I can't access my email. It keeps showing an e...,Subject: Assistance with Email Access\n\nDear ...,Subject: Assistance with Email Access\n\nDear ...
1,Tengo problemas con la VPN. No puedo conectarm...,Spanish,I am having problems with the VPN. I cannot co...,Subject: Assistance with VPN Connection Issues...,```Spanish\nAsunto: Asistencia con problemas d...
2,Mon imprimante ne répond pas et n'imprime plus...,French,"The customer message in English is: ""My printe...",Subject: Assistance with Your Printer Issue\n\...,```French\nObjet : Assistance concernant votre...
3,我无法访问公司的网站。每次都显示错误信息。请帮忙解决。,Chinese,I cannot access the company's website. It alwa...,"Dear Customer,\n\nThank you for reaching out t...",```Chinese\n亲爱的客户，\n\n感谢您与我们联系。很抱歉听到您在访问我们的网站时...


Branching and Merging Chains with LCEL

The idea here is to have multiple branching LLM Chains which work independently in parallel and then we merge their outputs finally using a merge LLM chain at the end to get a consolidated output

In [21]:
description_prompt =  ChatPromptTemplate.from_template(
    """Generate a two line description for the given topic:
      {topic}
""")

description_chain = (
    description_prompt
        |
    chatgpt
        |
    StrOutputParser()
)

In [22]:
pro_prompt = ChatPromptTemplate.from_template(
    """Generate three bullet points talking about the pros for the given topic:
      {topic}
""")

pro_chain = (
    pro_prompt
        |
    chatgpt
        |
    StrOutputParser()
)

In [23]:
con_prompt = ChatPromptTemplate.from_template(
    """Generate three bullet points talking about the cons for the given topic:
      {topic}
""")

con_chain = (
    con_prompt
        |
    chatgpt
        |
    StrOutputParser()
)

In [27]:
# Here we do not want to run chains in sequential order. we wnat to run chain in paraller, so we will use RunnableParallel to run chain in paraller 
# and to pass inputs to all the chain at a time, it will be done by itemgetter.

from operator import itemgetter
from langchain_core.runnables import RunnableParallel, RunnablePassthrough
branch_chain = (
    RunnableParallel(
        topic=itemgetter('topic'),
        description=description_chain,
        pros=pro_chain,
        cons=con_chain,
    )
)

In [28]:
branch_chain.invoke({"topic": "Generative AI"})

{'topic': 'Generative AI',
 'description': 'Generative AI refers to advanced algorithms that create new content, such as text, images, and music, by learning patterns from existing data. This technology has transformative applications across various fields, including art, entertainment, and business.',
 'pros': '- **Enhanced Creativity and Innovation**: Generative AI can assist artists, writers, and designers by providing new ideas, concepts, and variations, thereby fostering creativity and enabling the exploration of uncharted territories in various fields.\n\n- **Efficiency and Automation**: By automating repetitive tasks and generating content at scale, generative AI can significantly reduce the time and effort required for content creation, allowing professionals to focus on higher-level strategic work and decision-making.\n\n- **Personalization and Customization**: Generative AI can analyze user data to create highly personalized experiences, from tailored marketing messages to cu

In [29]:
merge_prompt = ChatPromptTemplate.from_template(
    """Create a report about {topic} with the following information:
      Description:
      {description}
      Pros:
      {pros}
      Cons:
      {cons}

      Report should be in the following format:

      Topic: <name of the topic>

      Description: <description of the topic>

      Pros and Cons:

      <table with two columns showing the 3 pros and cons of the topic>
""")

merge_chain = (
    merge_prompt
        |
    chatgpt
        |
    StrOutputParser()
)

In [30]:
final_chain = (
    branch_chain
      |
    merge_chain
)

In [31]:
response = final_chain.invoke({"topic": "Generative AI"})

In [32]:
from IPython.display import Markdown, display

display(Markdown(response))

# Report on Generative AI

**Topic:** Generative AI

**Description:**  
Generative AI refers to advanced algorithms that create new content, such as text, images, and music, by learning patterns from existing data. This technology has transformative applications across various fields, including art, entertainment, and business.

**Pros and Cons:**

| **Pros**                                           | **Cons**                                           |
|---------------------------------------------------|---------------------------------------------------|
| **Enhanced Creativity and Innovation**: Generative AI can produce unique content, designs, and solutions that may not be easily conceived by humans, fostering creativity and driving innovation across various fields such as art, music, and product development. | **Ethical Concerns**: Generative AI can produce content that may perpetuate biases, misinformation, or harmful stereotypes, raising ethical questions about accountability and the potential for misuse in various applications, such as deepfakes or propaganda. |
| **Increased Efficiency and Productivity**: By automating repetitive tasks and generating high-quality outputs quickly, generative AI can significantly reduce the time and effort required for content creation, allowing professionals to focus on more strategic and complex aspects of their work. | **Intellectual Property Issues**: The use of generative AI in creating art, music, or written content can lead to disputes over copyright and ownership, as it becomes challenging to determine the rights of the original creators versus those of the AI systems that generate derivative works. |
| **Personalization and Customization**: Generative AI can analyze user data to create tailored experiences, products, or services that meet individual preferences, enhancing customer satisfaction and engagement in industries like marketing, entertainment, and e-commerce. | **Job Displacement**: As generative AI becomes more capable, there is a growing concern that it could displace jobs in creative fields, such as writing, design, and music, leading to economic disruption and a need for workers to adapt to new roles in an increasingly automated landscape. |

Routing Chains with LCEL

The idea here is to have multiple individual LLM Chains which can perform their own tasks like summarize, sentiment etc.

We also have a router chain which can classify the user prompt intent and then route the user prompt to the relevant LLM Chain e.g if the user wants to summarize an article, his prompt request would be routed to the summarize chain automatically to get the result

In [33]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

classifier_prompt = ChatPromptTemplate.from_template(
        """Given the user instructions below for analyzing customer review,
           classify it as only one of the following categories:
            - summarize
            - sentiment
            - email

          Do not respond with more than one word.

          Instructions:
          {instruction}
""")

classifier_chain = (
    classifier_prompt
        |
    chatgpt
        |
    StrOutputParser()
)

In [34]:
summary_prompt = ChatPromptTemplate.from_template(
    """Act as a customer review analyst, given the following customer review,
       generate a short summary (max 2 lines) of the review.

       Customer Review:
       {review}
""")

summary_chain = (
    summary_prompt
        |
    chatgpt
        |
    StrOutputParser()
)

In [35]:
sentiment_prompt = ChatPromptTemplate.from_template(
    """Act as a customer review analyst, given the following customer review,
       find out the sentiment of the review.
       The sentiment can be either positive, negative or neutral.
       Return the result as a single word.

       Customer Review:
       {review}
""")

sentiment_chain = (
    sentiment_prompt
        |
    chatgpt
        |
    StrOutputParser()
)

In [36]:
email_prompt = ChatPromptTemplate.from_template(
    """Act as a customer review analyst,
    given the following customer review and its sentiment
    generate an email response to the customer based on the following conditions.
     - If the sentiment is positive or neutral thank them for their review
     - If the sentiment is negative, apologize to them

    Customer Review:
    {review}
    Sentiment:
    {sentiment}
""")

email_chain = (
    email_prompt
        |
    chatgpt
        |
    StrOutputParser()
)

In [37]:
def default_answer(query):
  return "Sorry instructions are not the defined intents"

In [38]:
from langchain_core.runnables import RunnableBranch

branch = RunnableBranch(
    (lambda x: "summarize" in x["topic"].lower(), summary_chain),
    (lambda x: "sentiment" in x["topic"].lower(), sentiment_chain),
    (lambda x: "email" in x["topic"].lower(), email_chain),
    default_answer,
)

In [47]:
full_chain = ({
                "topic": classifier_chain,
                "instruction": lambda input_prompt: input_prompt.get("instruction"),
                "review": lambda input_prompt: input_prompt.get("review"),
                "sentiment": lambda input_prompt: input_prompt.get("sentiment"),

              }
                  |
                branch)

In [42]:
sample_review = """
    Required a stylish lamp for my office space, and this particular one
    came with added storage at a reasonable price.
    The delivery was surprisingly quick, arriving within just two days.
    However, the pull string for the lamp suffered damage during transit.
    To my relief, the company promptly dispatched a replacement,
    which arrived within a few days. Assembly was a breeze.
    Then, I encountered an issue with a missing component,
    but their support team responded swiftly and provided the missing part.
    It appears to be a commendable company that genuinely values its
    customers and the quality of its products.
"""

In [43]:
summary = full_chain.invoke({"instruction": "Generate a summary for the given review","review": sample_review})
print(summary)

The customer praised the stylish lamp with added storage and quick delivery, despite initial issues with a damaged pull string and a missing component, which were promptly resolved by the supportive company.


In [44]:
sentiment = full_chain.invoke({"instruction": "Find out the sentiment of the customer in the review",
                   "review": sample_review})
print(sentiment)

Positive


In [45]:
response = full_chain.invoke({"instruction": "Write an email for the given customer review",
                              "review": sample_review,
                              "sentiment": sentiment})

In [46]:
print(response)

Subject: Thank You for Your Review!

Dear [Customer's Name],

Thank you so much for taking the time to share your experience with us! We’re thrilled to hear that you found the lamp stylish and that it met your needs for your office space. It’s great to know that our team was able to assist you promptly with the replacement and the missing component.

Your feedback is invaluable to us, and we truly appreciate your kind words about our commitment to customer satisfaction. If you have any further questions or need assistance in the future, please don’t hesitate to reach out.

Thank you once again for choosing us!

Best regards,

[Your Name]  
[Your Position]  
[Company Name]  
[Contact Information]  
